# Gemma Runtime Smoke-Test Harness

This notebook tests Google Gemma models across three runtimes:
- **Transformers** (Hugging Face, in-notebook Python)
- **Ollama** (external CLI/API)
- **llamafile** (single-file distribution)

## Goals
1. Confirm model runnability across different hardware configurations
2. Collect reliable, comparable timings
3. Provide clear debug signals for failures
4. Generate actionable recommendations for local/sovereign agentic apps

## Models Tested
- Gemma 3: 1b, 4b, 12b (with Gemma 2 fallbacks)
- All models are instruction-tuned variants (-it)

## Output
- `artifacts/results.jsonl` - Raw test results
- `artifacts/results.csv` - Flattened summary
- `artifacts/report.md` - OpenAI-generated analysis
- `artifacts/report.json` - Machine-readable report

## Setup: Imports and Environment

In [5]:
# ============================================================================
# CPU AND THREAD GUARDRAILS
# ============================================================================
# Set thread limits BEFORE importing torch/transformers to prevent CPU thrashing

import os
import psutil

# Limit to 4 threads or CPU count, whichever is smaller
CPU_THREADS = min(4, psutil.cpu_count() or 4)
os.environ['OMP_NUM_THREADS'] = str(CPU_THREADS)
os.environ['MKL_NUM_THREADS'] = str(CPU_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(CPU_THREADS)
os.environ['NUMEXPR_NUM_THREADS'] = str(CPU_THREADS)

print(f"🔧 Applied CPU thread limits: {CPU_THREADS} threads")
print(f"   OMP_NUM_THREADS={os.environ['OMP_NUM_THREADS']}")
print(f"   MKL_NUM_THREADS={os.environ['MKL_NUM_THREADS']}")
print(f"   OPENBLAS_NUM_THREADS={os.environ['OPENBLAS_NUM_THREADS']}")

# ============================================================================
# STANDARD IMPORTS
# ============================================================================

import sys
import json
import time
import subprocess
import platform
import warnings
import signal
import resource
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
from getpass import getpass
import pandas as pd

warnings.filterwarnings('ignore')

# Create output directory
ARTIFACTS_DIR = Path('artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

print(f"\nPython: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"CPU Count: {psutil.cpu_count()} logical, {psutil.cpu_count(logical=False) or 'N/A'} physical")
print(f"RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")


Python: 3.12.1 (main, Nov 27 2025, 10:47:52) [GCC 13.3.0]
Platform: Linux-6.8.0-1030-azure-x86_64-with-glibc2.39
CPU Count: 8
RAM: 31.3 GB


## Configuration Section

All user-configurable parameters are defined here.

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Runtime flags - which tests to run
RUN_TRANSFORMERS = True
RUN_OLLAMA = True
RUN_LLAMAFILE = True
RUN_OPENAI_REPORT = True

# Model lists
TRANSFORMERS_MODELS = [
    "google/gemma-3-1b-it",
    "google/gemma-3-4b-it",
    # "google/gemma-3-12b-it",
]

TRANSFORMERS_FALLBACK_MODELS = [
    # "google/gemma-2-2b-it",
    # "google/gemma-2-9b-it",
]

OLLAMA_MODELS = [
    "gemma3:1b",
    "gemma3:4b",
    # "gemma 3:12b",
]

OLLAMA_FALLBACK_MODELS = [
    "gemma2:2b",
    "gemma2:9b",
]

LLAMAFILE_MODELS = [
    "mozilla-ai/gemma-3-1b-it-llamafile",
    "mozilla-ai/gemma-3-4b-it-llamafile",
    # "mozilla-ai/gemma-3-12b-it-llamafile",
]

LLAMAFILE_FALLBACK_MODELS = [
    # "mozilla-ai/gemma-2-2b-it-llamafile",
]

# Timeouts (in seconds)
DOWNLOAD_TIMEOUT_S = 1800  # 30 minutes for large models
FIRST_LOAD_TIMEOUT_S = 600  # 10 minutes for first load
INFERENCE_TIMEOUT_S = 120  # 2 minutes per prompt
PROCESS_SHUTDOWN_TIMEOUT_S = 30  # 30 seconds for cleanup

# Generation parameters
MAX_TOKENS = 512
TEMPERATURE = 0.7
SEED = 42

# Prompts
SHORT_PROMPT = "What is the capital of France?"

COMPLEX_PROMPT = """Extract information from the following text and output as JSON with keys 'name', 'age', 'city', and 'occupation':

Text: John Smith is a 35-year-old software engineer living in San Francisco.

Output the result as valid JSON only, no additional text."""

PROMPTS = [
    {"name": "short", "text": SHORT_PROMPT},
    {"name": "complex", "text": COMPLEX_PROMPT},
]

# OpenAI configuration
OPENAI_MODEL = "gpt-4o"  # Use gpt-4o or latest available

# Output paths
RESULTS_JSONL = ARTIFACTS_DIR / "results.jsonl"
RESULTS_CSV = ARTIFACTS_DIR / "results.csv"
REPORT_MD = ARTIFACTS_DIR / "report.md"
REPORT_JSON = ARTIFACTS_DIR / "report.json"

print("✓ Configuration loaded")

✓ Configuration loaded


## Token Management

Prompt for required API tokens if not set in environment.

In [7]:
# Hugging Face token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN and RUN_TRANSFORMERS:
    print("Hugging Face token not found in environment.")
    HF_TOKEN = getpass("Enter HF_TOKEN (or press Enter to skip Transformers tests): ")
    if not HF_TOKEN:
        print("⚠ No HF token provided. Transformers tests will be skipped if gated models are encountered.")
        RUN_TRANSFORMERS = False
    else:
        os.environ['HF_TOKEN'] = HF_TOKEN
        print("✓ HF token set")

# OpenAI API key (always prompt, don't use env)
OPENAI_API_KEY = None
if RUN_OPENAI_REPORT:
    OPENAI_API_KEY = getpass("Enter OPENAI_API_KEY (or press Enter to skip report generation): ")
    if not OPENAI_API_KEY:
        print("⚠ No OpenAI API key provided. Report generation will be skipped.")
        RUN_OPENAI_REPORT = False
    else:
        print("✓ OpenAI API key set")

Hugging Face token not found in environment.
✓ HF token set
✓ OpenAI API key set


## Utility Functions

In [8]:
# ============================================================================
# STREAMING SUBPROCESS HELPER
# ============================================================================

def run_subprocess_with_streaming(
    cmd: List[str],
    timeout_s: int,
    heartbeat_interval_s: int = 30,
    description: str = ""
) -> Tuple[str, str, int, float]:
    """
    Run a subprocess with streaming output and heartbeat logging.
    
    Args:
        cmd: Command and arguments to run
        timeout_s: Maximum time to wait for process
        heartbeat_interval_s: Print heartbeat if no output for this many seconds
        description: Human-readable description for logging
    
    Returns:
        (stdout, stderr, returncode, elapsed_time)
    """
    import select
    
    start_time = time.time()
    last_output_time = start_time
    stdout_lines = []
    stderr_lines = []
    
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Starting: {description or ' '.join(cmd)}")
    
    try:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,  # Line buffered
        )
        
        # Non-blocking read loop
        while True:
            current_time = time.time()
            elapsed = current_time - start_time
            
            # Check timeout
            if elapsed > timeout_s:
                print(f"\n[{datetime.now().strftime('%H:%M:%S')}] ⏱️ TIMEOUT after {elapsed:.1f}s - terminating process")
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    print(f"[{datetime.now().strftime('%H:%M:%S')}] ⚠️ Process did not terminate cleanly")
                break
            
            # Check if process is still running
            if process.poll() is not None:
                # Process finished - read remaining output
                for line in process.stdout:
                    line = line.rstrip()
                    if line:
                        print(f"[{datetime.now().strftime('%H:%M:%S')}] {line}")
                        stdout_lines.append(line)
                for line in process.stderr:
                    line = line.rstrip()
                    if line:
                        print(f"[{datetime.now().strftime('%H:%M:%S')}] STDERR: {line}")
                        stderr_lines.append(line)
                break
            
            # Try to read output (non-blocking on Unix)
            try:
                # Use select on Unix systems for non-blocking I/O
                if hasattr(select, 'select'):
                    readable, _, _ = select.select([process.stdout, process.stderr], [], [], 0.5)
                    
                    if process.stdout in readable:
                        line = process.stdout.readline()
                        if line:
                            line = line.rstrip()
                            print(f"[{datetime.now().strftime('%H:%M:%S')}] {line}")
                            stdout_lines.append(line)
                            last_output_time = current_time
                    
                    if process.stderr in readable:
                        line = process.stderr.readline()
                        if line:
                            line = line.rstrip()
                            print(f"[{datetime.now().strftime('%H:%M:%S')}] STDERR: {line}")
                            stderr_lines.append(line)
                            last_output_time = current_time
                else:
                    # Fallback for Windows - just wait
                    time.sleep(0.5)
            except Exception as e:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Error reading output: {e}")
                time.sleep(0.5)
            
            # Print heartbeat if no output for a while
            if current_time - last_output_time >= heartbeat_interval_s:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] 💓 [heartbeat] still waiting for {description} — {elapsed:.0f}s elapsed, no new output for {current_time - last_output_time:.0f}s")
                last_output_time = current_time  # Reset to avoid spam
        
        elapsed_time = time.time() - start_time
        returncode = process.returncode if process.returncode is not None else -1
        
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ✓ Process completed in {elapsed_time:.1f}s (exit code: {returncode})")
        
        return ('\n'.join(stdout_lines), '\n'.join(stderr_lines), returncode, elapsed_time)
        
    except Exception as e:
        elapsed_time = time.time() - start_time
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ❌ Exception: {e}")
        try:
            process.terminate()
            process.wait(timeout=10)
        except:
            pass
        return ('', str(e), -1, elapsed_time)


# ============================================================================
# MEMORY GUARDRAILS
# ============================================================================

# Model size estimates in GB (fp32 / fp16)
MODEL_SIZE_ESTIMATES = {
    '1b': {'fp32': 2.5, 'fp16': 1.3},
    '2b': {'fp32': 5.0, 'fp16': 2.5},
    '4b': {'fp32': 9.0, 'fp16': 4.5},
    '9b': {'fp32': 20.0, 'fp16': 10.0},
    '12b': {'fp32': 27.0, 'fp16': 13.5},
}

def estimate_model_size(model_id: str) -> float:
    """Estimate model memory footprint in GB (assumes fp16)."""
    model_lower = model_id.lower()
    
    # Check sizes from largest to smallest to avoid substring matches
    for size_key in ['12b', '9b', '4b', '2b', '1b']:
        if size_key in model_lower:
            return MODEL_SIZE_ESTIMATES[size_key]['fp16']
    
    # Default to 2GB for unknown models
    return 2.0

def check_resources(model_id: str) -> Tuple[bool, str]:
    """
    Check if system has enough resources to load a model.
    
    Returns:
        (is_ok, message)
    """
    vm = psutil.virtual_memory()
    swap = psutil.swap_memory()
    
    available_gb = vm.available / (1024**3)
    total_gb = vm.total / (1024**3)
    swap_pct = swap.percent
    
    estimated_size_gb = estimate_model_size(model_id)
    
    # Check if model would use > 80% of available RAM
    threshold_gb = available_gb * 0.8
    
    print(f"\n{'='*60}")
    print(f"Resource Check: {model_id}")
    print(f"{'='*60}")
    print(f"Estimated model size: {estimated_size_gb:.1f} GB (fp16)")
    print(f"Available RAM: {available_gb:.1f} GB / {total_gb:.1f} GB total")
    print(f"Swap usage: {swap_pct:.1f}%")
    print(f"Threshold (80% of available): {threshold_gb:.1f} GB")
    
    # Check swap first
    if swap_pct > 50:
        msg = f"⚠️ WARNING: System is already swapping ({swap_pct:.1f}% swap used)"
        print(msg)
        print(f"{'='*60}\n")
        return False, msg
    
    # Check if model would fit
    if estimated_size_gb > threshold_gb:
        msg = f"🛑 BLOCKED: Model size ({estimated_size_gb:.1f} GB) exceeds 80% of available RAM ({threshold_gb:.1f} GB)"
        print(msg)
        print(f"{'='*60}\n")
        return False, msg
    
    # All good
    msg = f"✅ Resources OK: {estimated_size_gb:.1f} GB needed, {available_gb:.1f} GB available"
    print(msg)
    print(f"{'='*60}\n")
    return True, msg


# ============================================================================
# EXISTING UTILITY FUNCTIONS
# ============================================================================

def get_environment_fingerprint() -> Dict[str, Any]:
    """Collect environment metadata for reproducibility."""
    fingerprint = {
        "timestamp": datetime.now().isoformat(),
        "os": platform.platform(),
        "python_version": sys.version,
        "cpu_count": psutil.cpu_count(),
        "ram_gb": psutil.virtual_memory().total / (1024**3),
    }
    
    # Try to get package versions
    try:
        import torch
        fingerprint["torch_version"] = torch.__version__
        fingerprint["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            fingerprint["cuda_version"] = torch.version.cuda
            fingerprint["gpu_name"] = torch.cuda.get_device_name(0)
        if hasattr(torch.backends, 'mps'):
            fingerprint["mps_available"] = torch.backends.mps.is_available()
    except ImportError:
        fingerprint["torch_version"] = "not_installed"
    
    try:
        import transformers
        fingerprint["transformers_version"] = transformers.__version__
    except ImportError:
        fingerprint["transformers_version"] = "not_installed"
    
    return fingerprint


def log_phase(phase: str, **kwargs):
    """Log a timestamped phase marker."""
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "phase": phase,
        **kwargs
    }
    print(json.dumps(log_entry))
    return log_entry


def get_memory_info() -> Dict[str, float]:
    """Get current memory usage."""
    vm = psutil.virtual_memory()
    return {
        "total_gb": vm.total / (1024**3),
        "available_gb": vm.available / (1024**3),
        "used_gb": vm.used / (1024**3),
        "percent": vm.percent,
    }


def clear_memory():
    """Aggressively clear memory."""
    log_phase("CLEAR_START", memory=get_memory_info())
    
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except ImportError:
        pass
    
    import gc
    gc.collect()
    
    log_phase("CLEAR_DONE", memory=get_memory_info())


def save_result(result: Dict[str, Any]):
    """Append result to JSONL file."""
    with open(RESULTS_JSONL, 'a') as f:
        f.write(json.dumps(result) + '\n')


def create_result_record(
    runtime: str,
    model: str,
    status: str,
    prompt_name: str = "",
    prompt_text: str = "",
    output: str = "",
    duration_s: float = 0.0,
    device: str = "",
    error: str = "",
    **kwargs
) -> Dict[str, Any]:
    """Create a standardized result record."""
    return {
        "timestamp": datetime.now().isoformat(),
        "runtime": runtime,
        "model": model,
        "status": status,  # SUCCESS, FAILED, SKIPPED
        "prompt_name": prompt_name,
        "prompt_text": prompt_text,
        "output": output,
        "duration_s": duration_s,
        "device": device,
        "error": error,
        **kwargs
    }

print("✓ Utility functions loaded")

✓ Utility functions loaded


## Environment Fingerprint

In [9]:
env_fingerprint = get_environment_fingerprint()
print(json.dumps(env_fingerprint, indent=2))

{
  "timestamp": "2026-02-07T17:15:24.661743",
  "os": "Linux-6.8.0-1030-azure-x86_64-with-glibc2.39",
  "python_version": "3.12.1 (main, Nov 27 2025, 10:47:52) [GCC 13.3.0]",
  "cpu_count": 8,
  "ram_gb": 31.3463134765625,
  "torch_version": "2.10.0+cu128",
  "cuda_available": false,
  "mps_available": false,
  "transformers_version": "5.1.0"
}


## System Diagnostics

Detailed system resource analysis to understand performance constraints.


In [ ]:
# ============================================================================
# SYSTEM DIAGNOSTICS
# ============================================================================

print("\n" + "="*80)
print("SYSTEM DIAGNOSTICS")
print("="*80 + "\n")

diagnostics = {}

# Physical vs logical cores
try:
    diagnostics['cpu_logical_cores'] = psutil.cpu_count(logical=True)
    diagnostics['cpu_physical_cores'] = psutil.cpu_count(logical=False) or 'N/A'
except Exception as e:
    diagnostics['cpu_cores'] = f'Error: {e}'

# CPU frequency
try:
    freq = psutil.cpu_freq()
    if freq:
        diagnostics['cpu_freq_current_mhz'] = f"{freq.current:.0f}"
        diagnostics['cpu_freq_min_mhz'] = f"{freq.min:.0f}" if freq.min else 'N/A'
        diagnostics['cpu_freq_max_mhz'] = f"{freq.max:.0f}" if freq.max else 'N/A'
    else:
        diagnostics['cpu_freq'] = 'N/A'
except Exception as e:
    diagnostics['cpu_freq'] = f'Error: {e}'

# RAM
try:
    vm = psutil.virtual_memory()
    diagnostics['ram_total_gb'] = f"{vm.total / (1024**3):.1f}"
    diagnostics['ram_available_gb'] = f"{vm.available / (1024**3):.1f}"
    diagnostics['ram_used_pct'] = f"{vm.percent:.1f}%"
except Exception as e:
    diagnostics['ram'] = f'Error: {e}'

# Swap
try:
    swap = psutil.swap_memory()
    diagnostics['swap_total_gb'] = f"{swap.total / (1024**3):.1f}"
    diagnostics['swap_used_gb'] = f"{swap.used / (1024**3):.1f}"
    diagnostics['swap_free_gb'] = f"{swap.free / (1024**3):.1f}"
    diagnostics['swap_used_pct'] = f"{swap.percent:.1f}%"
except Exception as e:
    diagnostics['swap'] = f'Error: {e}'

# Disk I/O throughput estimate (100MB sequential read)
try:
    import tempfile
    test_size = 100 * 1024 * 1024  # 100MB
    test_file = tempfile.NamedTemporaryFile(delete=False)
    
    # Write test
    write_start = time.time()
    test_file.write(os.urandom(test_size))
    test_file.flush()
    os.fsync(test_file.fileno())
    write_time = time.time() - write_start
    
    test_file.close()
    
    # Read test
    read_start = time.time()
    with open(test_file.name, 'rb') as f:
        _ = f.read()
    read_time = time.time() - read_start
    
    os.unlink(test_file.name)
    
    write_throughput = test_size / (1024**2) / write_time  # MB/s
    read_throughput = test_size / (1024**2) / read_time  # MB/s
    
    diagnostics['disk_write_mb_s'] = f"{write_throughput:.0f}"
    diagnostics['disk_read_mb_s'] = f"{read_throughput:.0f}"
except Exception as e:
    diagnostics['disk_io'] = f'Error: {e}'

# Memory bandwidth estimate (512MB memcpy)
try:
    test_size = 512 * 1024 * 1024  # 512MB
    data = bytearray(test_size)
    
    copy_start = time.time()
    copy_data = bytearray(data)
    copy_time = time.time() - copy_start
    
    bandwidth_gb_s = test_size / (1024**3) / copy_time
    diagnostics['memory_bandwidth_gb_s'] = f"{bandwidth_gb_s:.2f}"
    
    # Cleanup
    del data, copy_data
except Exception as e:
    diagnostics['memory_bandwidth'] = f'Error: {e}'

# GPU memory (if CUDA)
try:
    import torch
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        diagnostics['gpu_memory_free_gb'] = f"{free / (1024**3):.1f}"
        diagnostics['gpu_memory_total_gb'] = f"{total / (1024**3):.1f}"
    else:
        diagnostics['gpu_memory'] = 'N/A (no CUDA)'
except Exception as e:
    diagnostics['gpu_memory'] = 'N/A'

# CPU governor / scaling
try:
    governor_file = Path('/sys/devices/system/cpu/cpu0/cpufreq/scaling_governor')
    if governor_file.exists():
        diagnostics['cpu_governor'] = governor_file.read_text().strip()
    else:
        diagnostics['cpu_governor'] = 'N/A'
except Exception as e:
    diagnostics['cpu_governor'] = 'N/A'

# Container memory limit
try:
    # Try cgroup v2
    mem_max = Path('/sys/fs/cgroup/memory.max')
    if mem_max.exists():
        limit = mem_max.read_text().strip()
        if limit != 'max':
            diagnostics['container_mem_limit_gb'] = f"{int(limit) / (1024**3):.1f}"
        else:
            diagnostics['container_mem_limit'] = 'unlimited'
    else:
        # Try cgroup v1
        mem_limit = Path('/sys/fs/cgroup/memory/memory.limit_in_bytes')
        if mem_limit.exists():
            limit = int(mem_limit.read_text().strip())
            if limit < 9223372036854771712:  # Not unlimited
                diagnostics['container_mem_limit_gb'] = f"{limit / (1024**3):.1f}"
            else:
                diagnostics['container_mem_limit'] = 'unlimited'
        else:
            diagnostics['container_mem_limit'] = 'N/A'
except Exception as e:
    diagnostics['container_mem_limit'] = 'N/A'

# Container CPU limit
try:
    # Try cgroup v2
    cpu_max = Path('/sys/fs/cgroup/cpu.max')
    if cpu_max.exists():
        content = cpu_max.read_text().strip()
        if content != 'max':
            parts = content.split()
            if len(parts) == 2:
                quota, period = parts
                if quota != 'max':
                    diagnostics['container_cpu_limit'] = f"{int(quota)/int(period):.2f} cores"
                else:
                    diagnostics['container_cpu_limit'] = 'unlimited'
            else:
                diagnostics['container_cpu_limit'] = 'unlimited'
        else:
            diagnostics['container_cpu_limit'] = 'unlimited'
    else:
        # Try cgroup v1
        cpu_quota = Path('/sys/fs/cgroup/cpu/cpu.cfs_quota_us')
        cpu_period = Path('/sys/fs/cgroup/cpu/cpu.cfs_period_us')
        if cpu_quota.exists() and cpu_period.exists():
            quota = int(cpu_quota.read_text().strip())
            period = int(cpu_period.read_text().strip())
            if quota > 0:
                diagnostics['container_cpu_limit'] = f"{quota/period:.2f} cores"
            else:
                diagnostics['container_cpu_limit'] = 'unlimited'
        else:
            diagnostics['container_cpu_limit'] = 'N/A'
except Exception as e:
    diagnostics['container_cpu_limit'] = 'N/A'

# Thermal throttling
try:
    thermal_zones = list(Path('/sys/class/thermal').glob('thermal_zone*/temp'))
    if thermal_zones:
        temps = []
        for zone in thermal_zones[:4]:  # Only check first 4
            try:
                temp = int(zone.read_text().strip()) / 1000.0
                temps.append(f"{temp:.0f}°C")
            except:
                pass
        diagnostics['thermal_zones'] = ', '.join(temps) if temps else 'N/A'
    else:
        diagnostics['thermal_zones'] = 'N/A'
except Exception as e:
    diagnostics['thermal_zones'] = 'N/A'

# ulimit -v (virtual memory limit)
try:
    soft, hard = resource.getrlimit(resource.RLIMIT_AS)
    if soft == resource.RLIM_INFINITY:
        diagnostics['ulimit_virtual_memory'] = 'unlimited'
    else:
        diagnostics['ulimit_virtual_memory_gb'] = f"{soft / (1024**3):.1f}"
except Exception as e:
    diagnostics['ulimit_virtual_memory'] = 'N/A'

# OOM scorer score
try:
    oom_score_file = Path(f'/proc/{os.getpid()}/oom_score')
    if oom_score_file.exists():
        diagnostics['oom_score'] = oom_score_file.read_text().strip()
    else:
        diagnostics['oom_score'] = 'N/A'
except Exception as e:
    diagnostics['oom_score'] = 'N/A'

# Print formatted table
print(f"{'Diagnostic':<35} {'Value':<30}")
print("-" * 65)
for key, value in diagnostics.items():
    # Format key (convert snake_case to Title Case)
    display_key = key.replace('_', ' ').title()
    print(f"{display_key:<35} {value:<30}")

print("\n" + "="*80 + "\n")


## Transformers Runtime Tests

Test models using Hugging Face Transformers library.

**Success criteria:**
- Model loads successfully
- Inference completes within timeout
- Output is generated for each prompt
- Device info is logged (CPU, CUDA, MPS)

In [10]:
def test_transformers_model(model_id: str) -> List[Dict[str, Any]]:
    """Test a single Transformers model with all prompts."""
    results = []
    model = None
    tokenizer = None
    
    # Check if model can fit in memory
    is_ok, resource_msg = check_resources(model_id)
    if not is_ok:
        results.append(create_result_record(
            runtime="transformers",
            model=model_id,
            status="SKIPPED",
            error="Insufficient resources: model estimated to exceed available memory",
        ))
        return results
    
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        
        # Determine device
        if torch.cuda.is_available():
            device = "cuda"
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"
        
        log_phase("LOAD_START", model=model_id, device=device, memory=get_memory_info())
        
        # Load model
        print(f"Loading {model_id}...")
        start_time = time.time()
        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=HF_TOKEN,
                torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                device_map="auto" if device == "cuda" else None,
            )
            if device != "cuda":
                model = model.to(device)
            model.eval()
            load_duration = time.time() - start_time
            log_phase("LOAD_DONE", model=model_id, duration_s=load_duration, memory=get_memory_info())
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)[:200]}"
            log_phase("LOAD_FAILED", model=model_id, error=error_msg)
            results.append(create_result_record(
                runtime="transformers",
                model=model_id,
                status="FAILED",
                error=error_msg,
                device=device,
            ))
            return results
        
        # Run inference for each prompt
        for prompt_info in PROMPTS:
            prompt_name = prompt_info["name"]
            prompt_text = prompt_info["text"]
            
            try:
                print(f"Running inference on {model_id} with prompt '{prompt_name}'...")
                log_phase("INFER_START", model=model_id, prompt=prompt_name)
                
                # Capture memory before inference
                mem_before = get_memory_info()
                mem_before_mb = mem_before['used_gb'] * 1024
                
                start_time = time.time()
                inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
                
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=MAX_TOKENS,
                        temperature=TEMPERATURE,
                        do_sample=True,
                    )
                
                output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                duration = time.time() - start_time
                
                # Capture memory after inference
                mem_after = get_memory_info()
                mem_after_mb = mem_after['used_gb'] * 1024
                mem_delta_mb = mem_after_mb - mem_before_mb
                
                log_phase("INFER_DONE", model=model_id, prompt=prompt_name, duration_s=duration)
                
                results.append(create_result_record(
                    runtime="transformers",
                    model=model_id,
                    status="SUCCESS",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    output=output_text,
                    duration_s=duration,
                    device=device,
                    mem_before_mb=mem_before_mb,
                    mem_after_mb=mem_after_mb,
                    mem_delta_mb=mem_delta_mb,
                ))
            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)[:200]}"
                log_phase("INFER_FAILED", model=model_id, prompt=prompt_name, error=error_msg)
                results.append(create_result_record(
                    runtime="transformers",
                    model=model_id,
                    status="FAILED",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    error=error_msg,
                    device=device,
                ))
    
    except ImportError as e:
        error_msg = f"Import error: {str(e)}"
        results.append(create_result_record(
            runtime="transformers",
            model=model_id,
            status="SKIPPED",
            error=error_msg,
        ))
    finally:
        # Cleanup
        del model
        del tokenizer
        clear_memory()
    
    return results


if RUN_TRANSFORMERS:
    print("\n" + "="*80)
    print("TRANSFORMERS RUNTIME TESTS")
    print("="*80 + "\n")
    
    # Try primary models first
    all_models = TRANSFORMERS_MODELS + TRANSFORMERS_FALLBACK_MODELS
    
    for model_id in all_models:
        print(f"\nTesting: {model_id}")
        results = test_transformers_model(model_id)
        for result in results:
            save_result(result)
            print(f"  {result['status']}: {result.get('prompt_name', 'N/A')} ({result.get('duration_s', 0):.2f}s)")
        print()
else:
    print("⚠ Transformers tests skipped")



TRANSFORMERS RUNTIME TESTS


Testing: google/gemma-3-1b-it
{"timestamp": "2026-02-07T17:15:30.129540", "phase": "LOAD_START", "model": "google/gemma-3-1b-it", "device": "cpu", "memory": {"total_gb": 31.3463134765625, "available_gb": 25.885440826416016, "used_gb": 5.460872650146484, "percent": 17.4}}


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

{"timestamp": "2026-02-07T17:19:36.486489", "phase": "LOAD_DONE", "model": "google/gemma-3-4b-it", "duration_s": 126.04803991317749, "memory": {"total_gb": 31.3463134765625, "available_gb": 6.907581329345703, "used_gb": 24.438732147216797, "percent": 78.0}}
{"timestamp": "2026-02-07T17:19:36.488271", "phase": "INFER_START", "model": "google/gemma-3-4b-it", "prompt": "short"}
{"timestamp": "2026-02-07T17:24:50.047676", "phase": "INFER_DONE", "model": "google/gemma-3-4b-it", "prompt": "short", "duration_s": 313.5593683719635}
{"timestamp": "2026-02-07T17:24:50.047817", "phase": "INFER_START", "model": "google/gemma-3-4b-it", "prompt": "complex"}
{"timestamp": "2026-02-07T17:25:19.891321", "phase": "INFER_DONE", "model": "google/gemma-3-4b-it", "prompt": "complex", "duration_s": 29.843475818634033}
{"timestamp": "2026-02-07T17:25:21.055443", "phase": "CLEAR_START", "memory": {"total_gb": 31.3463134765625, "available_gb": 22.618587493896484, "used_gb": 8.727725982666016, "percent": 27.8}}


## Ollama Runtime Tests

Test models using Ollama CLI.

**Success criteria:**
- Ollama is installed and server is reachable
- Model pulls successfully (or is already available)
- Inference completes within timeout
- Output is captured correctly

In [ ]:
def check_ollama_available() -> Tuple[bool, str]:
    """Check if Ollama is installed and server is running."""
    try:
        result = subprocess.run(
            ["ollama", "list"],
            capture_output=True,
            text=True,
            timeout=10,
        )
        if result.returncode == 0:
            return True, "Ollama is available"
        else:
            return False, f"Ollama CLI error: {result.stderr}"
    except FileNotFoundError:
        return False, "Ollama CLI not found"
    except subprocess.TimeoutExpired:
        return False, "Ollama CLI timeout"
    except Exception as e:
        return False, f"Ollama check error: {str(e)}"


def test_ollama_model(model_tag: str) -> List[Dict[str, Any]]:
    """Test a single Ollama model with all prompts."""
    results = []
    
    # Check if model can fit in memory
    is_ok, resource_msg = check_resources(model_tag)
    if not is_ok:
        results.append(create_result_record(
            runtime="ollama",
            model=model_tag,
            status="SKIPPED",
            error=resource_msg,
        ))
        return results
    
    try:
        # Pull model if needed
        print(f"Pulling {model_tag}... (timeout: {DOWNLOAD_TIMEOUT_S}s)")
        log_phase("DOWNLOAD_START", model=model_tag, runtime="ollama")
        
        stdout, stderr, returncode, elapsed_time = run_subprocess_with_streaming(
            cmd=["ollama", "pull", model_tag],
            description=f"ollama pull {model_tag}",
            timeout_s=DOWNLOAD_TIMEOUT_S,
        )
        
        if returncode != 0:
            error_msg = f"Pull failed: {stderr[:200]}"
            log_phase("DOWNLOAD_FAILED", model=model_tag, error=error_msg)
            results.append(create_result_record(
                runtime="ollama",
                model=model_tag,
                status="FAILED",
                error=error_msg,
            ))
            return results
        
        log_phase("DOWNLOAD_DONE", model=model_tag)
        
        # Run inference for each prompt
        for prompt_info in PROMPTS:
            prompt_name = prompt_info["name"]
            prompt_text = prompt_info["text"]
            
            try:
                print(f"Running inference on {model_tag} with prompt '{prompt_name}'...")
                log_phase("INFER_START", model=model_tag, prompt=prompt_name)
                
                # Capture memory before inference
                mem_before = get_memory_info()
                mem_before_mb = mem_before['used_gb'] * 1024
                
                start_time = time.time()
                stdout, stderr, returncode, elapsed_time = run_subprocess_with_streaming(
                    cmd=["ollama", "run", model_tag, prompt_text],
                    description=f"ollama run {model_tag}",
                    timeout_s=INFERENCE_TIMEOUT_S,
                )
                duration = time.time() - start_time
                
                # Capture memory after inference
                mem_after = get_memory_info()
                mem_after_mb = mem_after['used_gb'] * 1024
                mem_delta_mb = mem_after_mb - mem_before_mb
                
                if returncode == 0:
                    output_text = output_text.strip()
                    log_phase("INFER_DONE", model=model_tag, prompt=prompt_name, duration_s=duration)
                    
                    results.append(create_result_record(
                        runtime="ollama",
                        model=model_tag,
                        status="SUCCESS",
                        prompt_name=prompt_name,
                        prompt_text=prompt_text,
                        output=stdout,
                        duration_s=duration,
                        mem_before_mb=mem_before_mb,
                        mem_after_mb=mem_after_mb,
                        mem_delta_mb=mem_delta_mb,
                    ))
                else:
                    error_msg = f"Inference failed: {error[:200]}"
                    log_phase("INFER_FAILED", model=model_tag, prompt=prompt_name, error=error_msg)
                    results.append(create_result_record(
                        runtime="ollama",
                        model=model_tag,
                        status="FAILED",
                        prompt_name=prompt_name,
                        prompt_text=prompt_text,
                        error=error_msg,
                    ))
            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)[:200]}"
                log_phase("INFER_FAILED", model=model_tag, prompt=prompt_name, error=error_msg)
                results.append(create_result_record(
                    runtime="ollama",
                    model=model_tag,
                    status="FAILED",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    error=error_msg,
                ))
    except Exception as e:
        error_msg = f"{type(e).__name__}: {str(e)[:200]}"
        results.append(create_result_record(
            runtime="ollama",
            model=model_tag,
            status="FAILED",
            error=error_msg,
        ))
    finally:
        clear_memory()
    
    return results


if RUN_OLLAMA:
    print("\n" + "="*80)
    print("OLLAMA RUNTIME TESTS")
    print("="*80 + "\n")
    
    available, message = check_ollama_available()
    print(f"Ollama status: {message}\n")
    
    if available:
        all_models = OLLAMA_MODELS + OLLAMA_FALLBACK_MODELS
        
        for model_tag in all_models:
            print(f"\nTesting: {model_tag}")
            results = test_ollama_model(model_tag)
            for result in results:
                save_result(result)
                print(f"  {result['status']}: {result.get('prompt_name', 'N/A')} ({result.get('duration_s', 0):.2f}s)")
            print()
    else:
        print(f"⚠ Ollama tests skipped: {message}")
        # Save skipped record
        for model_tag in OLLAMA_MODELS:
            save_result(create_result_record(
                runtime="ollama",
                model=model_tag,
                status="SKIPPED",
                error=message,
            ))
else:
    print("⚠ Ollama tests skipped (disabled in config)")


## llamafile Runtime Tests

Test models using llamafile executables.

**Success criteria:**
- llamafile downloads successfully
- Executable permissions are set correctly
- Model runs and produces output
- Process terminates cleanly

In [ ]:
def download_llamafile(repo_id: str) -> Tuple[Optional[Path], str]:
    """Download a llamafile from Hugging Face."""
    try:
        from huggingface_hub import hf_hub_download
        
        # Extract model name from repo_id
        model_name = repo_id.split('/')[-1]
        
        # Look for .llamafile in the repo
        log_phase("DOWNLOAD_START", model=repo_id, runtime="llamafile")
        
        print(f"Downloading {repo_id}... (timeout: {DOWNLOAD_TIMEOUT_S}s)")
        
        try:
            # Try to download the llamafile
            llamafile_path = hf_hub_download(
                repo_id=repo_id,
                filename=f"{model_name}.llamafile",
                token=HF_TOKEN,
                cache_dir=str(ARTIFACTS_DIR / "llamafile_cache"),
            )
            
            # Make executable
            import stat
            path = Path(llamafile_path)
            path.chmod(path.stat().st_mode | stat.S_IEXEC)
            
            log_phase("DOWNLOAD_DONE", model=repo_id)
            return path, ""
        except Exception as e:
            error_msg = f"Download failed: {str(e)[:200]}"
            log_phase("DOWNLOAD_FAILED", model=repo_id, error=error_msg)
            return None, error_msg
    except ImportError:
        return None, "huggingface_hub not installed"


def test_llamafile_model(repo_id: str) -> List[Dict[str, Any]]:
    """Test a single llamafile model with all prompts."""
    results = []
    
    # Check if model can fit in memory
    is_ok, resource_msg = check_resources(repo_id)
    if not is_ok:
        results.append(create_result_record(
            runtime="llamafile",
            model=repo_id,
            status="SKIPPED",
            error="Insufficient resources: model estimated to exceed available memory",
        ))
        return results
    
    # Download llamafile
    llamafile_path, error = download_llamafile(repo_id)
    if not llamafile_path:
        results.append(create_result_record(
            runtime="llamafile",
            model=repo_id,
            status="FAILED",
            error=error,
        ))
        return results
    
    # Test each prompt
    for prompt_info in PROMPTS:
        prompt_name = prompt_info["name"]
        prompt_text = prompt_info["text"]
        
        try:
            print(f"Running inference on {repo_id} with prompt '{prompt_name}'...")
            log_phase("INFER_START", model=repo_id, prompt=prompt_name)
            
            # Capture memory before inference
            mem_before = get_memory_info()
            mem_before_mb = mem_before['used_gb'] * 1024
            
            start_time = time.time()
            # Run llamafile with prompt
            stdout, stderr, returncode, elapsed_time = run_subprocess_with_streaming(
                cmd=[str(llamafile_path), "-p", prompt_text, "--temp", str(TEMPERATURE), "-n", str(MAX_TOKENS)],
                description=f"llamafile inference for {repo_id}",
                timeout_s=INFERENCE_TIMEOUT_S,
            )
            duration = time.time() - start_time
            
            # Capture memory after inference
            mem_after = get_memory_info()
            mem_after_mb = mem_after['used_gb'] * 1024
            mem_delta_mb = mem_after_mb - mem_before_mb
            
            if returncode == 0:
                log_phase("INFER_DONE", model=repo_id, prompt=prompt_name, duration_s=duration)
                
                results.append(create_result_record(
                    runtime="llamafile",
                    model=repo_id,
                    status="SUCCESS",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    output=stdout,
                    duration_s=duration,
                    mem_before_mb=mem_before_mb,
                    mem_after_mb=mem_after_mb,
                    mem_delta_mb=mem_delta_mb,
                ))
            else:
                error_msg = f"Execution failed: {error[:200]}"
                log_phase("INFER_FAILED", model=repo_id, prompt=prompt_name, error=error_msg)
                results.append(create_result_record(
                    runtime="llamafile",
                    model=repo_id,
                    status="FAILED",
                    prompt_name=prompt_name,
                    prompt_text=prompt_text,
                    error=error_msg,
                ))
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)[:200]}"
            log_phase("INFER_FAILED", model=repo_id, prompt=prompt_name, error=error_msg)
            results.append(create_result_record(
                runtime="llamafile",
                model=repo_id,
                status="FAILED",
                prompt_name=prompt_name,
                prompt_text=prompt_text,
                error=error_msg,
            ))
        finally:
            clear_memory()
    
    return results


if RUN_LLAMAFILE:
    print("\n" + "="*80)
    print("LLAMAFILE RUNTIME TESTS")
    print("="*80 + "\n")
    
    all_models = LLAMAFILE_MODELS + LLAMAFILE_FALLBACK_MODELS
    
    for repo_id in all_models:
        print(f"\nTesting: {repo_id}")
        results = test_llamafile_model(repo_id)
        for result in results:
            save_result(result)
            print(f"  {result['status']}: {result.get('prompt_name', 'N/A')} ({result.get('duration_s', 0):.2f}s)")
        print()
else:
    print("⚠ llamafile tests skipped")


## Results Summary

Load and display test results.

In [ ]:
# Load results
if RESULTS_JSONL.exists():
    results_data = []
    with open(RESULTS_JSONL, 'r') as f:
        for line in f:
            results_data.append(json.loads(line))
    
    print(f"Loaded {len(results_data)} result records")
    
    # Create DataFrame
    df = pd.DataFrame(results_data)
    
    # Save CSV
    df.to_csv(RESULTS_CSV, index=False)
    print(f"✓ Saved results to {RESULTS_CSV}")
    
    # Display summary
    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80 + "\n")
    
    print("Status counts:")
    print(df['status'].value_counts())
    print()
    
    print("Results by runtime:")
    print(df.groupby(['runtime', 'status']).size())
    print()
    
    if len(df[df['status'] == 'SUCCESS']) > 0:
        print("Successful runs - average duration:")
        success_df = df[df['status'] == 'SUCCESS']
        print(success_df.groupby(['runtime', 'model'])['duration_s'].mean())
else:
    print("⚠ No results file found")


## OpenAI Report Generation

Generate an analysis report using OpenAI API.

**Success criteria:**
- Report analyzes which models were runnable
- Failures are clustered by type
- Practical recommendations are provided
- Best latency metrics are highlighted

In [ ]:
if RUN_OPENAI_REPORT and RESULTS_JSONL.exists():
    print("\n" + "="*80)
    print("GENERATING OPENAI REPORT")
    print("="*80 + "\n")
    
    try:
        from openai import OpenAI
        
        # Load all results
        results_data = []
        with open(RESULTS_JSONL, 'r') as f:
            for line in f:
                results_data.append(json.loads(line))
        
        # Estimate token count
        prompt_est_tokens = len(json.dumps(results_data)) // 4
        print(f"\nPreparing OpenAI API call...")
        print(f"  Sending {len(results_data)} results (~{prompt_est_tokens} tokens)")
        print(f"  Model: {OPENAI_MODEL}")
        print(f"  This may take 30-60s...")
        
        # Prepare prompt for OpenAI
        prompt = f"""Analyze the following benchmark results from testing Google Gemma models across three runtimes (Transformers, Ollama, llamafile).

Environment:
{json.dumps(env_fingerprint, indent=2)}

Test Results:
{json.dumps(results_data, indent=2)}

Please provide a concise report (max 1000 words) that includes:

1. Executive Summary: Which models were runnable in this environment and which runtime(s) worked best?

2. Failure Analysis: Cluster failures by type (e.g., download/gating issues, OOM, runtime not installed, timeouts) and explain the root causes.

3. Performance Metrics: Create a table showing the best achieved latency (duration_s) and estimated tokens/sec for each successfully tested model and runtime.

4. Recommendations: For someone building "local, sovereign agentic apps" on this machine class:
   - Which runtime should they use?
   - Which model sizes are realistic?
   - What are the practical tradeoffs?

Format the response in Markdown."""
        
        print("Calling OpenAI API...")
        
        try:
            import httpx
            timeout_client = OpenAI(
                api_key=OPENAI_API_KEY,
                timeout=httpx.Timeout(120.0, connect=30.0)
            )
            
            response = timeout_client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {"role": "system", "content": "You are an expert in ML infrastructure and model deployment."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3,
            )
            
            # Print usage stats
            if hasattr(response, 'usage') and response.usage:
                print(f"\n✓ OpenAI API call successful!")
                print(f"  Prompt tokens: {response.usage.prompt_tokens}")
                print(f"  Completion tokens: {response.usage.completion_tokens}")
                print(f"  Total tokens: {response.usage.total_tokens}")
        
        except Exception as e:
            error_type = type(e).__name__
            print(f"\n❌ OpenAI API error ({error_type}): {e}")
            if "timeout" in str(e).lower() or "TimeoutError" in error_type:
                print("The API call timed out after 120s. Try again later or check your connection.")
            elif "rate" in str(e).lower() or "RateLimitError" in error_type:
                print("Rate limit exceeded. Wait a few minutes before trying again.")
            elif "connection" in str(e).lower() or "ConnectionError" in error_type:
                print("Could not connect to OpenAI API. Check your internet connection.")
            raise
        
        report_text = response.choices[0].message.content
        
        # Save markdown report
        with open(REPORT_MD, 'w') as f:
            f.write(report_text)
        print(f"✓ Saved report to {REPORT_MD}")
        
        # Save JSON report
        report_json = {
            "generated_at": datetime.now().isoformat(),
            "model_used": OPENAI_MODEL,
            "environment": env_fingerprint,
            "report_text": report_text,
            "results_summary": {
                "total_tests": len(results_data),
                "successful": len([r for r in results_data if r['status'] == 'SUCCESS']),
                "failed": len([r for r in results_data if r['status'] == 'FAILED']),
                "skipped": len([r for r in results_data if r['status'] == 'SKIPPED']),
            }
        }
        
        with open(REPORT_JSON, 'w') as f:
            json.dump(report_json, f, indent=2)
        print(f"✓ Saved report metadata to {REPORT_JSON}")
        
        # Display report
        print("\n" + "="*80)
        print("REPORT")
        print("="*80 + "\n")
        print(report_text)
        
    except Exception as e:
        print(f"⚠ Report generation failed: {type(e).__name__}: {str(e)}")
elif not RUN_OPENAI_REPORT:
    print("⚠ OpenAI report generation skipped (disabled in config)")
else:
    print("⚠ No results file found - skipping report generation")


## Test Complete

All tests have been executed. Check the `artifacts/` directory for:
- `results.jsonl` - Raw test results
- `results.csv` - Flattened summary
- `report.md` - Human-readable analysis
- `report.json` - Machine-readable report metadata